In [1]:
import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

# Import Gyftakis's custom metrics and RankNet
from utils import custom_ap_scorer
try:
    from machine_learning.neuralnet.python.keras.ranknet import RankNet
except ModuleNotFoundError:
    print("ERROR: Ensure the 'machine_learning' folder is in your current directory!")

In [2]:
# ==========================================
# 1. LOAD & FIX DATA
# ==========================================
print("Loading train and test datasets...")
train_df = pd.read_csv("train_df.csv", index_col=0)
test_df = pd.read_csv("test_df.csv", index_col=0)

# --- THE CRITICAL FIX: INVERT TEST LABELS ONLY ---
print("Applying label inversion fix to test data...")
test_df['target'] = 1 - test_df['target']

feature_names = train_df.columns.tolist()
feature_names.remove('label')
feature_names.remove('target')

# Extract features and targets
X = train_df[feature_names].values
Y = train_df["target"].values

X_test = test_df[feature_names].values
Y_test = test_df["target"].values

print(f"Training on {X.shape[0]} nodes with {X.shape[1]} features.")

Loading train and test datasets...
Applying label inversion fix to test data...
Training on 1055 nodes with 181 features.


In [3]:
# ==========================================
# 2. TRAIN RANKNET
# ==========================================
# Using the best parameters from Gyftakis's original GridSearch
print("\nTraining RankNet (nhidden=64, l2=0.01)...")
start_time = time.time()

rank_net = RankNet(nhidden=64, l2=0.01)
rank_net.fit(X, Y)

print("Training took {:.1f} seconds".format(time.time() - start_time))


Training RankNet (nhidden=64, l2=0.01)...





Training took 50.1 seconds


In [4]:
# import gc
# from machine_learning.neuralnet.python.keras.ranknet import RankNet

# results = []
# start_time = time.time()
# for nhidden in [10, 32, 64, 128]:
#     for l2 in [0, 0.01, 0.1, 1]:
#         print(f"Using nhidden={nhidden}, l2={l2}")
#         model = RankNet(nhidden=nhidden, l2=l2)
#         cv_results = cross_validate(model, X, Y, cv=5, scoring=sklearn_custom_ap_scorer)
#         results.append(((nhidden, l2), cv_results['test_score']))
#         del model
#         gc.collect()
#         print("Done")
#     tf.keras.backend.clear_session()
        
# print("GridSearch took {:.1f} seconds".format(time.time() - start_time))
# best_parameters, best_scores = sorted(results, key=lambda x: np.median(x[1]), reverse=True)[0]
# print("Best parameters={}".format(best_parameters))
# print("Best score={}".format(np.median(best_scores)))

In [5]:
# ==========================================
# 3. EVALUATION
# ==========================================
print("\nPredicting and Scoring...")
Y_rank_net_predict = rank_net.predict(X_test)

# Calculate Average Precision using the exact logic from utils.py
ap_score = custom_ap_scorer(Y_test, Y_rank_net_predict)

print("\n" + "="*50)
print("FINAL RANKNET AVERAGE PRECISION: {:.4f}".format(ap_score))
print("="*50)


Predicting and Scoring...
3/3 [==============================] - 0s 4ms/step

FINAL RANKNET AVERAGE PRECISION: 0.9286
